In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect(r'C:\Users\marzieh\Documents\GitHub\Tennis-project\data\tennis.duckdb', read_only=True)
con.execute("SHOW TABLES").df()

,name
0,_build_info
1,_import_audit
2,_schema_audit
3,_source_files
4,game_point_by_point
5,match_away_score
6,match_away_team
7,match_event
8,match_home_score
9,match_home_team


In [3]:
query = """
    SELECT e.match_id,
           h.player_id as home_id,
           h.name as home_name, 
           a.player_id as away_id,
           a.name as away_name, 
           winner_code
    FROM match_home_team as h
    INNER JOIN match_away_team as a
    ON h.match_id = a.match_id
    INNER JOIN match_event as e
    ON e.match_id = h.match_id
    WHERE winner_code IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY e.match_id
        ORDER BY e.match_id
    ) = 1
"""
df = con.execute(query).df()
df

,match_id,home_id,home_name,away_id,away_name,winner_code
0,12022685,280202,Sieg M.,111879,Fung S.,2
1,12023348,75813,Zhang Z.,420969,Kriznik M.,1
2,12018941,62000,Kan V.,284061,Ezzat Y.,1
3,12021328,445463,Tianmi Mi.,107429,Turati A.,2
4,12024249,213076,Erhard M.,330085,Popovic S.,1
...,...,...,...,...,...,...
9603,12211603,46428,Malečková J.,63238,Morderger Y.,1
9604,12210883,233060,Seggerman R.,210033,Glinka D.,1
9605,12211329,214247,Liu H.,285196,Shin W.,1
9606,12211900,26203,Gomez E.,121248,Roveri Sidney G.,2


In [7]:
winner_names = pd.Series(
    df["home_name"].where(df["winner_code"] == 1, df["away_name"])
)
winner_names.value_counts().head(1)

home_name
Popko D.    28
Name: count, dtype: int64